<a href="https://colab.research.google.com/github/aliyu-zy/html-azy.portfolio/blob/main/Aliyu_Yahaya_Capstone_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from google.colab import files
import pickle # to save tokenizer

# 1. UPLOAD DATASET
# Your CSV must have 2 columns: 'message' and 'label'
# label: 0 = Ham/Safe, 1 = Fraud/Spam
print("Upload your fraud_sms_dataset.csv")
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])
print("Data Loaded:")
print(df.head())
print("\nClass Distribution:")
print(df['label'].value_counts()) # Check if imbalanced

Upload your fraud_sms_dataset.csv


Saving fraud_sms_dataset 2.csv to fraud_sms_dataset 2.csv
Data Loaded:
                                                text  label
0  EKEDC / IKEDC Alert: Your electricity prepaid ...  fraud
1  LCredit / PalmPay Alert: Your emergency loan a...  fraud
2  Your One-Time Password (OTP) for online bankin...  legit
3  The meeting has been rescheduled to tomorrow m...  legit
4  Dear subscriber, your monthly DSTV Compact sub...  legit

Class Distribution:
label
fraud    230
legit    229
Fraud      1
Name: count, dtype: int64


In [28]:
# 2. PREPROCESS TEXT DATA
X = df['text'].astype(str) # SMS text
y = df['label'].str.lower() # Convert labels to lowercase to handle inconsistencies
y = y.map({'legit': 0, 'fraud': 1}).astype(int).values # Convert string labels to numerical (0 or 1) and then to int numpy array

# Convert text to numbers using Tokenizer
MAX_WORDS = 5000 # Only use top 5000 words
MAX_LEN = 100 # Max length of each SMS

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>") # <OOV> = Out of Vocabulary
tokenizer.fit_on_texts(X)
sequences = tokenizer.texts_to_sequences(X)

# Pad sequences so all are same length
X_pad = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Split into Train and Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y)

# Handle imbalanced classes: fraud is usually <10% of data
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weight_dict)

# Save tokenizer so we can use it later in Gradio
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)


Class Weights: {0: np.float64(1.0043668122270741), 1: np.float64(0.9956709956709957)}


In [24]:
# 3. BUILD BIDIRECTIONAL LSTM MODEL
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN), # Turn words into vectors
    Bidirectional(LSTM(64, return_sequences=True)), # Read SMS forwards + backwards
    Bidirectional(LSTM(32)), # Get final summary
    Dropout(0.5), # Prevent overfitting
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # 1 output: probability of fraud
])

# Use Precision/Recall because accuracy is misleading on imbalanced data
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
# 4. TRAIN MODEL
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint('fraud_sms_model.keras', save_best_only=True, monitor='val_recall')

# Explicitly convert y_train and y_val to TensorFlow float32 tensors
y_train_tensor = tf.cast(y_train, tf.float32)
y_val_tensor = tf.cast(y_val, tf.float32)

history = model.fit(
    X_train, y_train_tensor,
    validation_data=(X_val, y_val_tensor),
    epochs=15,
    batch_size=32,
    class_weight=class_weight_dict, # Tell model to care more about fraud class
    callbacks=[early_stop, checkpoint]
)

# 5. SAVE FINAL MODEL + TOKENIZER
model.save('fraud_sms_model_final.keras')
files.download('fraud_sms_model_final.keras')
files.download('tokenizer.pkl')
print("Model and Tokenizer downloaded!")


Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 294ms/step - accuracy: 0.6658 - loss: 0.6710 - precision: 0.6462 - recall: 0.7405 - val_accuracy: 0.9022 - val_loss: 0.5797 - val_precision: 0.9744 - val_recall: 0.8261
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 217ms/step - accuracy: 0.9375 - loss: 0.4018 - precision: 0.9709 - recall: 0.9027 - val_accuracy: 0.6196 - val_loss: 0.6963 - val_precision: 0.5679 - val_recall: 1.0000
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 208ms/step - accuracy: 0.9511 - loss: 0.1623 - precision: 0.9718 - recall: 0.9297 - val_accuracy: 0.9565 - val_loss: 0.1168 - val_precision: 1.0000 - val_recall: 0.9130
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 373ms/step - accuracy: 0.9946 - loss: 0.0557 - precision: 0.9946 - recall: 0.9946 - val_accuracy: 0.9891 - val_loss: 0.0481 - val_precision: 0.9787 - val_recall: 1.0000
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 252ms/step - accuracy: 0.9946 - loss: 0.0349 - precision: 0.9893 - recall: 1.0000 - val_accuracy: 1.0000 - val_loss: 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model and Tokenizer downloaded!


In [30]:
!pip install -q gradio

import gradio as gr
import tensorflow as tf
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. LOAD MODEL AND TOKENIZER
model = tf.keras.models.load_model('fraud_sms_model_final.keras')
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

MAX_LEN = 100

# 2. PREDICTION FUNCTION
def predict_fraud_sms(message):
    """Takes SMS text and returns Fraud or Safe with confidence"""
    # Preprocess the input text same way as training
    seq = tokenizer.texts_to_sequences([message])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    # Get prediction probability
    pred_prob = model.predict(pad, verbose=0)[0][0]

    if pred_prob > 0.5:
        label = "FRAUD / SCAM"
        confidence = pred_prob * 100
        color = "🔴"
    else:
        label = "SAFE / HAM"
        confidence = (1 - pred_prob) * 100
        color = "🟢"

    return f"{color} Prediction: {label}", f"Confidence: {confidence:.2f}%"

# 3. BUILD GRADIO INTERFACE
iface = gr.Interface(
    fn=predict_fraud_sms,
    inputs=gr.Textbox(lines=4, placeholder="Paste SMS here... e.g. 'You won 1M! Click link'"),
    outputs=[gr.Textbox(label="Result"), gr.Textbox(label="Confidence")],
    title="Fraud SMS Classifier",
    description="Paste any SMS message to check if it's Fraud or Safe. Model: Bidirectional LSTM",
    examples=[
        ["CONGRAT! You have won N500,000. Dial *123# to claim"],
        ["Hi mom, can you send me 5k for data?"],
        ["Your bank account will be blocked. Verify here: bit.ly/fakebank"]
    ]
)

# 4. LAUNCH APP
iface.launch(share=True) # share=True gives you a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad5bdedc5b845145a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
